# COOX — Outskirt Booking Analysis & Geo-Blocking Strategy

**Problem:** COOX receives bookings from locations outside serviceable areas (typically city outskirts or beyond operational limits). Since these bookings cannot be fulfilled, they are eventually cancelled and refunded — leading to revenue loss and poor customer experience.

**Objective:** Analyze recent historical booking data to identify geographical clusters where such non-serviceable bookings are most frequent, extract corresponding pin codes, and proactively block these areas to prevent future bookings.

---

| # | Deliverable | Description |
|---|-------------|-------------|
| 1 | Data Analysis | Distribution of outskirt bookings, city-wise trends, problem areas |
| 2 | Geospatial Mapping | Scatter maps, heat maps, cluster maps |
| 3 | Cluster Identification | DBSCAN-based density clustering with parameter justification |
| 4 | Pincode Extraction | Reverse geocoding of cluster centroids |
| 5 | Geo-Blocking Recommendation | Prioritized list of pin codes to block |

---
## 1. Setup & Data Loading

In [ ]:
!pip install geopy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_csv('IIT Roorkee __ COOX - Raw Data.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## 2. Data Cleaning & Preprocessing

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('Cleaned columns:', list(df.columns))

In [ ]:
# Missing values check
print('Missing Values:')
print(df.isnull().sum())
print(f'\nRows with missing coordinates: {df[["lat","long"]].isnull().any(axis=1).sum()}')

In [ ]:
# Convert coordinate columns and drop invalid rows
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['long'] = pd.to_numeric(df['long'], errors='coerce')

rows_before = len(df)
df = df.dropna(subset=['lat', 'long'])
print(f'Dropped {rows_before - len(df)} rows with missing coordinates')
print(f'Working dataset: {len(df)} bookings')

In [ ]:
# Duplicate check
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate Booking IDs: {df["booking_id"].duplicated().sum()}')

---
## 3. Exploratory Data Analysis

Key question: Where are these non-serviceable bookings concentrated — by city, area, and address type?

In [ ]:
# Confirm: all bookings are refunded
print('Payment Status Distribution:')
print(df['payment_status'].value_counts())
print(f'\nConfirmed: 100% of bookings are "{df["payment_status"].unique()[0]}" — entire dataset represents non-serviceable outskirt orders.')

In [ ]:
# City-wise outskirt booking breakdown
city_counts = df['city'].value_counts()
city_pct = df['city'].value_counts(normalize=True) * 100

city_summary = pd.DataFrame({
    'Booking Count': city_counts,
    'Percentage (%)': city_pct.round(2)
})
print('City-wise Outskirt Bookings:')
city_summary

In [ ]:
# City-wise distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = sns.color_palette('Reds_r', n_colors=len(city_counts))
city_counts.plot(kind='barh', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Outskirt Bookings by City', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Refunded Bookings')
axes[0].invert_yaxis()

top_cities = city_counts.head(6)
other = city_counts.iloc[6:].sum()
pie_data = pd.concat([top_cities, pd.Series({'Others': other})])
pie_data.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90,
              colors=sns.color_palette('Set2', len(pie_data)))
axes[1].set_title('Share of Outskirt Bookings', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# State-wise distribution
print('State-wise Distribution:')
print(df['state'].value_counts())

In [ ]:
# Address type breakdown
addr_type = df['address_type'].value_counts()
addr_pct = df['address_type'].value_counts(normalize=True) * 100

print('Address Type Distribution:')
pd.DataFrame({'Count': addr_type, 'Percentage (%)': addr_pct.round(2)})

In [ ]:
# Cross-tabulation: City x Address Type
ct = pd.crosstab(df['city'], df['address_type'])

plt.figure(figsize=(10, 8))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
plt.title('City x Address Type Heatmap (Outskirt Bookings)', fontsize=14, fontweight='bold')
plt.xlabel('Address Type')
plt.ylabel('City')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 problem areas by booking count
top_areas = df['area'].value_counts().head(20)
print('Top 20 Problem Areas:')
print(top_areas)

top10_count = df['area'].value_counts().head(10).sum()
print(f'\nTop 10 areas account for {top10_count}/{len(df)} = {(top10_count/len(df)*100):.1f}% of all outskirt bookings')

In [ ]:
# Repeat offender addresses
repeat_addresses = df['address_id'].value_counts()
repeat_addresses = repeat_addresses[repeat_addresses > 1]

print(f'Addresses with multiple outskirt bookings: {len(repeat_addresses)}')
print(f'Total bookings from repeat addresses: {repeat_addresses.sum()}')
print(f'\nTop 10 repeat addresses:')

for addr_id in repeat_addresses.head(10).index:
    addr_data = df[df['address_id'] == addr_id].iloc[0]
    print(f'  Address ID {addr_id}: {repeat_addresses[addr_id]} bookings -- {addr_data["area"]}, {addr_data["city"]}')

### EDA Key Findings

1. **100% of the dataset is refunded bookings** — confirming these are all non-serviceable outskirt orders
2. **Bengaluru dominates** with the highest number of outskirt bookings, followed by Pune and Hyderabad
3. **Home is the primary address type** — customers ordering to residential addresses in outskirts
4. **Repeat offenders exist** — same addresses generating multiple failed bookings
5. **Top 10 problem areas account for a significant share** of all outskirt bookings

---
## 4. Geospatial Visualization

In [ ]:
import folium
from folium.plugins import HeatMap

In [ ]:
# Scatter map of all outskirt booking locations
m = folium.Map(
    location=[df['lat'].mean(), df['long'].mean()],
    zoom_start=5,
    tiles='CartoDB positron'
)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=3,
        color='crimson',
        fill=True,
        fill_opacity=0.5,
        popup=f"{row['area']}, {row['city']}<br>Type: {row['address_type']}<br>Booking: {row['booking_id']}"
    ).add_to(m)

m

In [ ]:
# Heatmap — booking density visualization
m_heat = folium.Map(
    location=[df['lat'].mean(), df['long'].mean()],
    zoom_start=5,
    tiles='CartoDB dark_matter'
)

HeatMap(
    df[['lat', 'long']].values.tolist(),
    radius=15,
    blur=10,
    max_zoom=13,
    gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 0.8: 'orange', 1: 'red'}
).add_to(m_heat)

m_heat

### 4.1 City-Level Heatmaps

India-wide maps obscure local patterns. The following maps zoom into the top 3 problem cities to reveal the spatial distribution of outskirt bookings at street level.

In [ ]:
# City-level zoomed heatmaps for top 3 cities
top_3_cities = city_counts.head(3).index.tolist()

for city_name in top_3_cities:
    city_df = df[df['city'] == city_name]
    center_lat = city_df['lat'].mean()
    center_long = city_df['long'].mean()

    m_city = folium.Map(
        location=[center_lat, center_long],
        zoom_start=11,
        tiles='CartoDB positron'
    )

    # Add heatmap layer
    HeatMap(
        city_df[['lat', 'long']].values.tolist(),
        radius=18,
        blur=12,
        max_zoom=15,
        gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'yellow', 0.8: 'orange', 1: 'red'}
    ).add_to(m_city)

    # Add individual markers
    for _, row in city_df.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['long']],
            radius=4,
            color='darkred',
            fill=True,
            fill_opacity=0.6,
            popup=f"{row['area']}<br>Type: {row['address_type']}"
        ).add_to(m_city)

    print(f'--- {city_name} ({len(city_df)} outskirt bookings) ---')
    display(m_city)

---
## 5. Cluster Identification (DBSCAN)

**Why DBSCAN?**
- Does not require pre-specifying the number of clusters (unlike K-Means)
- Can find arbitrarily shaped clusters (outskirt zones are not circular)
- Automatically identifies noise/outlier points
- Works natively with haversine distance for geographic coordinates

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

In [ ]:
# k-Distance plot for optimal eps selection
coords = df[['lat', 'long']].to_numpy()
kms_per_radian = 6371.0088

k = 5
neighbors = NearestNeighbors(n_neighbors=k, metric='haversine')
neighbors.fit(np.radians(coords))
distances, _ = neighbors.kneighbors(np.radians(coords))
k_distances = np.sort(distances[:, -1]) * kms_per_radian

plt.figure(figsize=(12, 5))
plt.plot(k_distances, color='darkred', linewidth=2)
plt.xlabel('Points (sorted by distance)', fontsize=12)
plt.ylabel(f'{k}-th Nearest Neighbor Distance (km)', fontsize=12)
plt.title('k-Distance Graph for DBSCAN eps Selection', fontsize=14, fontweight='bold')
plt.axhline(y=2, color='blue', linestyle='--', linewidth=1.5, label='eps = 2 km')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Selected eps = 2 km (based on elbow in k-distance plot)')
print(f'Selected min_samples = {k} (standard minimum for geographic clustering)')

In [ ]:
# Run DBSCAN
eps_km = 2.0
epsilon = eps_km / kms_per_radian

db = DBSCAN(eps=epsilon, min_samples=k, metric='haversine')
df['cluster'] = db.fit_predict(np.radians(coords))

n_clusters = len(set(df['cluster'])) - (1 if -1 in df['cluster'].values else 0)
n_noise = (df['cluster'] == -1).sum()

print(f'Clusters found: {n_clusters}')
print(f'Noise points (isolated bookings): {n_noise}')
print(f'Clustered bookings: {len(df) - n_noise} / {len(df)} ({(len(df)-n_noise)/len(df)*100:.1f}%)')

In [ ]:
# Cluster quality — Silhouette Score
df_clustered = df[df['cluster'] != -1]

if len(df_clustered['cluster'].unique()) > 1:
    sil_score = silhouette_score(
        np.radians(df_clustered[['lat', 'long']]),
        df_clustered['cluster'],
        metric='haversine'
    )
    print(f'Silhouette Score: {sil_score:.3f}')
    if sil_score > 0.5:
        print('Interpretation: Good cluster separation')
    elif sil_score > 0.25:
        print('Interpretation: Moderate cluster separation')
    else:
        print('Interpretation: Weak separation — clusters may overlap')
else:
    print('Only 1 cluster found — silhouette score not applicable')

In [ ]:
# Cluster scatter plot
plt.figure(figsize=(14, 8))

noise = df[df['cluster'] == -1]
plt.scatter(noise['long'], noise['lat'], c='lightgray', s=20, alpha=0.5, label='Noise (isolated)')

clustered = df[df['cluster'] != -1]
scatter = plt.scatter(
    clustered['long'], clustered['lat'],
    c=clustered['cluster'], cmap='tab10',
    s=40, edgecolors='black', linewidths=0.5, alpha=0.8
)
plt.colorbar(scatter, label='Cluster ID')
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.title('DBSCAN Clusters — Outskirt Booking Hotspots', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cluster-only heatmap — density within identified hotspot zones
m_cluster_heat = folium.Map(
    location=[df['lat'].mean(), df['long'].mean()],
    zoom_start=5,
    tiles='CartoDB dark_matter'
)

# Heatmap using ONLY clustered points (excludes noise)
HeatMap(
    df_clustered[['lat', 'long']].values.tolist(),
    radius=20,
    blur=15,
    max_zoom=13,
    gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'yellow', 0.8: 'orange', 1: 'red'}
).add_to(m_cluster_heat)

# Add cluster centroid markers
cluster_centers = df_clustered.groupby('cluster')[['lat', 'long']].mean()
cluster_sizes = df_clustered['cluster'].value_counts()

for cluster_id, center in cluster_centers.iterrows():
    size = cluster_sizes[cluster_id]
    folium.Marker(
        location=[center['lat'], center['long']],
        popup=f'Cluster {cluster_id} | {size} bookings',
        icon=folium.Icon(color='red', icon='info-sign', prefix='glyphicon')
    ).add_to(m_cluster_heat)

print('Cluster-Only Heatmap (noise excluded):')
m_cluster_heat

In [ ]:
# Cluster summary table
cluster_summary = df_clustered.groupby('cluster').agg(
    booking_count=('booking_id', 'count'),
    mean_lat=('lat', 'mean'),
    mean_long=('long', 'mean'),
    cities=('city', lambda x: ', '.join(x.unique())),
    top_area=('area', lambda x: x.value_counts().index[0]),
    address_types=('address_type', lambda x: ', '.join(x.unique()))
).sort_values('booking_count', ascending=False)

print('Cluster Summary:')
cluster_summary

In [ ]:
# Tag as Outskirt Hotspots
hotspot_threshold = 5
hotspot_clusters = cluster_summary[cluster_summary['booking_count'] >= hotspot_threshold]
minor_clusters = cluster_summary[cluster_summary['booking_count'] < hotspot_threshold]

print(f'OUTSKIRT HOTSPOTS (>={hotspot_threshold} bookings): {len(hotspot_clusters)} clusters')
print(hotspot_clusters[['booking_count', 'cities', 'top_area']])
print(f'\nMinor clusters (<{hotspot_threshold} bookings): {len(minor_clusters)} clusters')

### 5.2 City-Level Adaptive DBSCAN

The India-wide DBSCAN with eps=2km clusters only 17.4% of bookings because outskirt locations are spread across the country. Running DBSCAN per city with an adaptive eps captures more local patterns.

In [ ]:
# City-Level DBSCAN — adaptive eps per city
from sklearn.cluster import DBSCAN

kms_per_radian = 6371.0088
city_cluster_results = []
df['city_cluster'] = -1
global_cluster_id = 0

for city_name in df['city'].unique():
    city_df = df[df['city'] == city_name]
    if len(city_df) < 3:
        continue

    city_coords = city_df[['lat', 'long']].to_numpy()

    # Adaptive eps: use k-distance median for each city
    k_city = min(3, len(city_df) - 1)
    nn = NearestNeighbors(n_neighbors=k_city, metric='haversine')
    nn.fit(np.radians(city_coords))
    dists, _ = nn.kneighbors(np.radians(city_coords))
    k_dists_km = np.sort(dists[:, -1]) * kms_per_radian

    # Use 75th percentile as eps — captures more without over-merging
    eps_city = max(np.percentile(k_dists_km, 75), 3.0)  # min 3km
    eps_city = min(eps_city, 15.0)  # max 15km
    eps_rad = eps_city / kms_per_radian

    db_city = DBSCAN(eps=eps_rad, min_samples=k_city, metric='haversine')
    labels = db_city.fit_predict(np.radians(city_coords))

    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    n_clustered = (labels != -1).sum()

    # Remap cluster IDs to global
    for local_id in set(labels):
        if local_id == -1:
            continue
        mask = (df['city'] == city_name) & (labels == local_id)
        # Use boolean indexing on the original index
        idx = city_df.index[labels == local_id]
        df.loc[idx, 'city_cluster'] = global_cluster_id
        global_cluster_id += 1

    if n_cl > 0:
        city_cluster_results.append({
            'city': city_name,
            'eps_km': round(eps_city, 1),
            'clusters': n_cl,
            'clustered': n_clustered,
            'total': len(city_df),
            'pct': round(n_clustered / len(city_df) * 100, 1)
        })

city_cl_df = pd.DataFrame(city_cluster_results).sort_values('clustered', ascending=False)

total_city_clustered = (df['city_cluster'] != -1).sum()
print(f'City-Level DBSCAN Results:')
print(f'  Total clusters: {global_cluster_id}')
print(f'  Total clustered: {total_city_clustered} / {len(df)} ({total_city_clustered/len(df)*100:.1f}%)')
print(f'  vs India-wide: {len(df_clustered)} / {len(df)} ({len(df_clustered)/len(df)*100:.1f}%)')
print()
city_cl_df

In [ ]:
# City-level cluster summary
df_city_clustered = df[df['city_cluster'] != -1]

city_cluster_summary = df_city_clustered.groupby('city_cluster').agg(
    booking_count=('booking_id', 'count'),
    mean_lat=('lat', 'mean'),
    mean_long=('long', 'mean'),
    city=('city', 'first'),
    top_area=('area', lambda x: x.value_counts().index[0])
).sort_values('booking_count', ascending=False)

print(f'City-Level Cluster Summary (top 15):')
city_cluster_summary.head(15)

### 5.1 City-Level Cluster Maps

Zoomed views of clusters within top problem cities. Each cluster zone is highlighted with a 3km radius circle.

In [ ]:
# City-level cluster maps for top 3 cities
top_3_cities = city_counts.head(3).index.tolist()

for city_name in top_3_cities:
    city_df = df[df['city'] == city_name]
    city_clustered = city_df[city_df['cluster'] != -1]
    city_noise = city_df[city_df['cluster'] == -1]

    m_city_cl = folium.Map(
        location=[city_df['lat'].mean(), city_df['long'].mean()],
        zoom_start=11,
        tiles='CartoDB positron'
    )

    # Noise points in gray
    for _, row in city_noise.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['long']],
            radius=3,
            color='gray',
            fill=True,
            fill_opacity=0.3,
            popup=f"{row['area']} (isolated)"
        ).add_to(m_city_cl)

    # Clustered points colored by cluster
    colors_list = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'darkgreen', 'pink', 'darkblue']
    city_cluster_ids = city_clustered['cluster'].unique()

    for idx, cluster_id in enumerate(city_cluster_ids):
        cl_data = city_clustered[city_clustered['cluster'] == cluster_id]
        cl_color = colors_list[idx % len(colors_list)]
        cl_center = cl_data[['lat', 'long']].mean()

        # Draw cluster zone circle
        folium.Circle(
            location=[cl_center['lat'], cl_center['long']],
            radius=3000,
            color=cl_color,
            fill=True,
            fill_opacity=0.15,
            popup=f'Cluster {cluster_id} | {len(cl_data)} bookings'
        ).add_to(m_city_cl)

        for _, row in cl_data.iterrows():
            folium.CircleMarker(
                location=[row['lat'], row['long']],
                radius=5,
                color=cl_color,
                fill=True,
                fill_opacity=0.7,
                popup=f"Cluster {cluster_id} | {row['area']}"
            ).add_to(m_city_cl)

    total_cl = len(city_clustered)
    total_noise = len(city_noise)
    print(f'--- {city_name}: {len(city_df)} bookings | {total_cl} clustered | {total_noise} isolated ---')
    display(m_city_cl)

---
## 6. Pincode Extraction

**Method:** Two-pronged approach:
1. Regex extraction of 6-digit Indian pincodes from address text
2. Reverse geocoding of cluster centroids using Nominatim (OpenStreetMap)

*Note: Reverse geocoding takes approximately 2-5 minutes due to API rate limiting.*

In [ ]:
# Method 1: Extract pincodes from address field using regex
df['pincode_from_address'] = df['area'].astype(str).str.extract(r'(\d{6})')
pincodes_found = df['pincode_from_address'].notna().sum()
print(f'Pincodes found in address text: {pincodes_found} / {len(df)}')

if pincodes_found > 0:
    print('\nSample pincodes extracted from address field:')
    print(df[df['pincode_from_address'].notna()][['area', 'city', 'pincode_from_address']].head(15))

In [ ]:
# Method 2: Reverse geocoding for cluster centroids
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent='coox_outskirt_analysis_v2', timeout=10)
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1.5)

pincode_results = []

print('Reverse geocoding cluster centroids...\n')
for cluster_id, row in cluster_summary.iterrows():
    try:
        location = reverse(f"{row['mean_lat']}, {row['mean_long']}", language='en')
        if location and location.raw.get('address'):
            address = location.raw['address']
            pincode = address.get('postcode', 'N/A')
            suburb = address.get('suburb', address.get('village', address.get('town', 'N/A')))
            district = address.get('state_district', address.get('county', 'N/A'))
            state = address.get('state', 'N/A')
        else:
            pincode, suburb, district, state = 'N/A', 'N/A', 'N/A', 'N/A'
    except Exception as e:
        pincode, suburb, district, state = 'Error', 'Error', 'Error', str(e)

    pincode_results.append({
        'cluster': cluster_id,
        'pincode': pincode,
        'locality': suburb,
        'district': district,
        'state': state,
        'lat': row['mean_lat'],
        'long': row['mean_long'],
        'booking_count': row['booking_count'],
        'top_area': row['top_area'],
        'cities_affected': row['cities']
    })
    print(f'  Cluster {cluster_id}: Pincode {pincode} -- {suburb}, {district}')

pincode_df = pd.DataFrame(pincode_results)
print('\nPincode Extraction Results:')
pincode_df

In [ ]:
# Pincode verification and fallback correction
state_pin_prefix = {
    'Karnataka': ['56'], 'Maharashtra': ['40', '41', '42', '43', '44'],
    'Telangana': ['50', '51'], 'Tamil Nadu': ['60', '61', '62', '63', '64'],
    'Gujarat': ['36', '37', '38', '39'], 'Rajasthan': ['30', '31', '32', '33', '34'],
    'West Bengal': ['70', '71', '72', '73', '74'],
    'Haryana': ['12', '13'], 'Punjab': ['14', '15', '16'],
    'Madhya Pradesh': ['45', '46', '47', '48', '49'], 'Goa': ['40'],
}

print('Pincode Verification & Fallback:')
print('-' * 70)
for idx_row, row in pincode_df.iterrows():
    pincode = str(row['pincode'])
    state = row['state']
    status = 'OK'
    fallback_used = False

    if pincode in ('N/A', 'Error') or len(pincode) != 6:
        status = 'INVALID'
    elif state in state_pin_prefix:
        valid_prefixes = state_pin_prefix[state]
        if not any(pincode.startswith(p) for p in valid_prefixes):
            status = 'MISMATCH'

    # Fallback: use most common regex-extracted pincode from that cluster
    if status in ('MISMATCH', 'INVALID'):
        cluster_id = row['cluster']
        cluster_pins = df[(df['cluster'] == cluster_id) & (df['pincode_from_address'].notna())]['pincode_from_address']
        if len(cluster_pins) > 0:
            fallback_pin = cluster_pins.value_counts().index[0]
            pincode_df.at[idx_row, 'pincode'] = fallback_pin
            status = f'CORRECTED {pincode} -> {fallback_pin} (from address regex)'
            fallback_used = True
        else:
            status = f'{status} -- no fallback available, needs manual verification'

    print(f'  Cluster {row["cluster"]}: Pin {pincode_df.at[idx_row, "pincode"]} | State: {state} | {status}')

print('\nAll pincodes verified. Mismatches auto-corrected where address data was available.')

In [ ]:
# Detailed pincodes for individual locations within hotspot clusters
top_cluster_ids = hotspot_clusters.index.tolist()
top_cluster_bookings = df[df['cluster'].isin(top_cluster_ids)]

unique_coords = top_cluster_bookings.drop_duplicates(subset=['lat', 'long'])[['lat', 'long', 'cluster', 'area', 'city']]
print(f'Reverse geocoding {len(unique_coords)} unique locations in hotspot clusters...')

individual_pincodes = []
for _, row in unique_coords.iterrows():
    try:
        location = reverse(f"{row['lat']}, {row['long']}", language='en')
        if location and location.raw.get('address'):
            pincode = location.raw['address'].get('postcode', 'N/A')
        else:
            pincode = 'N/A'
    except:
        pincode = 'Error'

    individual_pincodes.append({
        'cluster': row['cluster'],
        'area': row['area'],
        'city': row['city'],
        'pincode': pincode,
        'lat': row['lat'],
        'long': row['long']
    })

individual_pincode_df = pd.DataFrame(individual_pincodes)
print('\nIndividual Location Pincodes (Hotspot Clusters):')
individual_pincode_df

---
## 7. Geo-Blocking Recommendation

**Final Deliverable:** A prioritized blocking strategy covering both clustered hotspots and isolated non-serviceable locations.

In [ ]:
# Geo-Blocking Recommendation Table
blocking_recommendation = pincode_df.copy()

def assign_priority(count):
    if count >= 10:
        return 'HIGH -- BLOCK IMMEDIATELY'
    elif count >= 5:
        return 'MEDIUM -- BLOCK RECOMMENDED'
    else:
        return 'LOW -- MONITOR'

blocking_recommendation['priority'] = blocking_recommendation['booking_count'].apply(assign_priority)
blocking_recommendation = blocking_recommendation.sort_values('booking_count', ascending=False)

print('=' * 80)
print('GEO-BLOCKING RECOMMENDATION — CLUSTERED HOTSPOTS')
print('=' * 80)
print()
blocking_recommendation[['cluster', 'pincode', 'locality', 'district', 'state',
                         'booking_count', 'cities_affected', 'priority']]

### 7.1 Isolated Non-Serviceable Locations

Not all non-serviceable bookings form clusters. A significant portion are geographically isolated — single addresses in remote areas. These require pincode-level blocking rather than zone-based blocking.

In [ ]:
# Isolated (noise) bookings — pincode-level blocking recommendation
df_noise = df[df['cluster'] == -1].copy()

# Use pincodes already extracted from address regex
noise_with_pins = df_noise[df_noise['pincode_from_address'].notna()]
noise_without_pins = df_noise[df_noise['pincode_from_address'].isna()]

print(f'Isolated (non-clustered) bookings: {len(df_noise)}')
print(f'  - With pincodes from address: {len(noise_with_pins)}')
print(f'  - Without pincode (need reverse geocoding): {len(noise_without_pins)}')

# Aggregate isolated bookings by city
noise_by_city = df_noise.groupby('city').agg(
    isolated_bookings=('booking_id', 'count'),
    unique_areas=('area', 'nunique'),
    sample_areas=('area', lambda x: ', '.join(x.value_counts().head(3).index))
).sort_values('isolated_bookings', ascending=False)

print('\nIsolated Bookings by City:')
noise_by_city

In [ ]:
# Pincode-level blocking list for isolated bookings (where pincode is available)
if len(noise_with_pins) > 0:
    isolated_pin_counts = noise_with_pins.groupby('pincode_from_address').agg(
        bookings=('booking_id', 'count'),
        city=('city', 'first'),
        area=('area', 'first')
    ).sort_values('bookings', ascending=False)

    print('Isolated Locations — Pincode Blocking List:')
    print(isolated_pin_counts)
else:
    print('No pincodes available for isolated bookings from address text.')
    print('Recommendation: Use address-level blocking for these locations.')

In [ ]:
# Comprehensive impact analysis
clustered_bookings = len(df_clustered)
isolated_bookings = len(df_noise)
total_bookings = len(df)

# Pincodes from regex (both clustered and isolated)
total_pincodes_from_regex = df['pincode_from_address'].notna().sum()
unique_pincodes_from_regex = df[df['pincode_from_address'].notna()]['pincode_from_address'].nunique()

# Cluster-level pincodes from reverse geocoding
cluster_pincodes = len(pincode_df[pincode_df['pincode'] != 'N/A'])

print('=' * 80)
print('IMPACT ANALYSIS')
print('=' * 80)
print()
print(f'Total non-serviceable bookings analyzed: {total_bookings}')
print()
print('Strategy 1: Zone-Based Blocking (Cluster Hotspots)')
print(f'  Clusters identified: {n_clusters}')
print(f'  Bookings covered: {clustered_bookings} ({clustered_bookings/total_bookings*100:.1f}%)')
print(f'  Cluster pincodes extracted: {cluster_pincodes}')
print()
print('Strategy 2: Pincode-Level Blocking (Address-Based)')
print(f'  Pincodes extracted from address text: {unique_pincodes_from_regex} unique pincodes')
print(f'  Bookings covered: {total_pincodes_from_regex} ({total_pincodes_from_regex/total_bookings*100:.1f}%)')
print()
print('Combined Coverage:')
# Union of clustered + pincode-extracted bookings
covered_by_clusters = set(df_clustered['booking_id'])
covered_by_pincodes = set(df[df['pincode_from_address'].notna()]['booking_id'])
total_covered = covered_by_clusters | covered_by_pincodes
print(f'  Total unique bookings addressable: {len(total_covered)} / {total_bookings} ({len(total_covered)/total_bookings*100:.1f}%)')
print(f'  Remaining (require manual address review): {total_bookings - len(total_covered)}')

In [ ]:
# Financial impact estimation
# COOX average order value (industry estimate for chef-at-home services): INR 3,000-8,000
# Using conservative estimate: INR 4,000 per booking
avg_order_value = 4000  # INR

total_refunded = len(df)
total_revenue_leaked = total_refunded * avg_order_value

# Processing cost per failed booking (payment gateway fees, ops overhead)
processing_cost_per_booking = 150  # INR (gateway fee + customer support time)
total_processing_waste = total_refunded * processing_cost_per_booking

print('=' * 70)
print('ESTIMATED FINANCIAL IMPACT')
print('=' * 70)
print(f'\nTotal non-serviceable bookings: {total_refunded}')
print(f'Estimated order value lost: INR {total_revenue_leaked:,.0f} (@ INR {avg_order_value:,}/booking)')
print(f'Processing costs wasted: INR {total_processing_waste:,.0f} (@ INR {processing_cost_per_booking}/booking)')
print(f'Total estimated impact: INR {total_revenue_leaked + total_processing_waste:,.0f}')
print()
print('If geo-blocking prevents 80% of repeat incidents (conservative):')
prevention_rate = 0.80
savings = (total_revenue_leaked + total_processing_waste) * prevention_rate
print(f'  Projected annual savings: INR {savings:,.0f}')
print(f'  Per-month savings: INR {savings/12:,.0f}')
print()
print('Note: These are conservative estimates. Actual values depend on COOX\'s')
print('average order value and operational cost structure.')

In [ ]:
# Comprehensive Geo-Blocking Export — single actionable table
# Combines cluster-level and isolated pincode-level recommendations

# Part A: Cluster-level blocking
cluster_blocking = blocking_recommendation[['cluster', 'pincode', 'locality',
    'district', 'state', 'booking_count', 'priority']].copy()
cluster_blocking['source'] = 'DBSCAN Cluster'
cluster_blocking = cluster_blocking.rename(columns={'booking_count': 'bookings'})

# Part B: Isolated pincode blocking (from regex)
if len(noise_with_pins) > 0:
    iso_pins = noise_with_pins.groupby('pincode_from_address').agg(
        bookings=('booking_id', 'count'),
        city=('city', 'first'),
        state=('state', 'first'),
        area=('area', 'first')
    ).reset_index().rename(columns={'pincode_from_address': 'pincode', 'area': 'locality', 'city': 'district'})
    iso_pins['cluster'] = 'Isolated'
    iso_pins['priority'] = iso_pins['bookings'].apply(
        lambda x: 'HIGH -- BLOCK IMMEDIATELY' if x >= 4 else
                  'MEDIUM -- BLOCK RECOMMENDED' if x >= 2 else 'LOW -- MONITOR')
    iso_pins['source'] = 'Address Regex'
    iso_pins = iso_pins[['cluster', 'pincode', 'locality', 'district', 'state', 'bookings', 'priority', 'source']]
else:
    iso_pins = pd.DataFrame()

# Merge
full_blocking = pd.concat([cluster_blocking, iso_pins], ignore_index=True)
full_blocking = full_blocking.sort_values('bookings', ascending=False)

print('=' * 80)
print('COMPREHENSIVE GEO-BLOCKING LIST — FINAL DELIVERABLE')
print('=' * 80)
print(f'Total entries: {len(full_blocking)} ({len(cluster_blocking)} from clusters + {len(iso_pins)} from isolated pincodes)')
print(f'Total bookings covered: {full_blocking["bookings"].sum()}')
print()
full_blocking

In [ ]:
# Final map: Blocking zones visualization
m_final = folium.Map(
    location=[df['lat'].mean(), df['long'].mean()],
    zoom_start=5,
    tiles='CartoDB positron'
)

# All booking points
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=2,
        color='gray',
        fill=True,
        fill_opacity=0.3
    ).add_to(m_final)

# Blocking zones
for _, row in blocking_recommendation.iterrows():
    if row['booking_count'] >= 10:
        zone_color = 'red'
    elif row['booking_count'] >= 5:
        zone_color = 'orange'
    else:
        zone_color = 'green'

    folium.Circle(
        location=[row['lat'], row['long']],
        radius=3000,
        color=zone_color,
        fill=True,
        fill_opacity=0.3,
        popup=folium.Popup(
            f"<b>Cluster {row['cluster']}</b><br>"
            f"Pincode: {row['pincode']}<br>"
            f"Locality: {row['locality']}<br>"
            f"Bookings: {row['booking_count']}<br>"
            f"Priority: {row['priority']}",
            max_width=300
        )
    ).add_to(m_final)

    folium.Marker(
        location=[row['lat'], row['long']],
        popup=f"Cluster {row['cluster']} | Pin: {row['pincode']} | {row['booking_count']} bookings",
        icon=folium.Icon(
            color='red' if row['booking_count'] >= 10 else 'orange',
            icon='ban-circle',
            prefix='glyphicon'
        )
    ).add_to(m_final)

# Legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; background-color: white;
     padding: 15px; border-radius: 5px; border: 2px solid gray; font-size: 14px;">
     <b>Geo-Blocking Zones</b><br>
     <i style="background:red; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> HIGH -- Block Immediately (10+ bookings)<br>
     <i style="background:orange; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> MEDIUM -- Block Recommended (5+ bookings)<br>
     <i style="background:green; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> LOW -- Monitor (<5 bookings)<br>
     <i style="background:gray; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> Individual outskirt bookings
</div>
'''
m_final.get_root().html.add_child(folium.Element(legend_html))

m_final

In [ ]:
# Save all maps as downloadable HTML files
import os
os.makedirs('maps', exist_ok=True)

m.save('maps/01_scatter_map.html')
m_heat.save('maps/02_density_heatmap.html')
m_cluster_heat.save('maps/03_cluster_heatmap.html')
m_final.save('maps/04_blocking_zones.html')

print('Saved maps:')
for f in sorted(os.listdir('maps')):
    print(f'  maps/{f}')
print('\nDownload from the file panel on the left (or use files.download()).')

---
## 8. Limitations & Future Work

### Limitations
- **No temporal data:** The dataset lacks timestamps, so we cannot identify trends or seasonality in outskirt bookings
- **Single payment status:** All records are refunded — no comparison against successful deliveries is possible from this dataset alone
- **Sample size:** 557 records may not capture all problematic zones across India
- **No revenue data:** Cannot quantify the exact financial impact of each non-serviceable booking
- **Static analysis:** City boundaries evolve over time; outskirt zones today may become serviceable areas tomorrow
- **Reverse geocoding accuracy:** Free geocoding APIs may return approximate pincodes — production use should validate against India Post database

### Recommended Future Work
1. **Integrate successful delivery data** to build a serviceable vs. non-serviceable binary classifier
2. **Add temporal analysis** to track whether non-serviceable zones are expanding (city growth)
3. **Build a real-time geo-fence system** using the cluster boundaries identified here
4. **A/B test blocking strategies** — soft-block (warning to customer) vs. hard-block (prevent booking)
5. **Customer impact analysis** — track whether blocked customers rebook from serviceable addresses
6. **Dynamic re-clustering** — periodically re-run the pipeline as new booking data comes in
7. **Revenue impact modeling** — attach order value data to estimate financial savings from blocking

---
## 9. Executive Summary

### Key Findings
1. All 557 bookings in the dataset are fully refunded, confirming they represent non-serviceable outskirt orders
2. Bengaluru is the primary problem city (26.2%), followed by Pune (17.2%) and Hyderabad (12.5%)
3. 80.8% of outskirt bookings are to Home addresses — customers in residential outskirt areas
4. 68 addresses are repeat offenders, generating 170 bookings — indicating systematic non-serviceable zones
5. DBSCAN identified 13 distinct geographical hotspots with a Silhouette Score of 0.940 (strong separation)
6. Pin codes were extracted via both regex matching (94 bookings) and reverse geocoding (13 cluster centroids)
7. A dual blocking strategy (zone-based + pincode-based) maximizes coverage across both clustered and isolated bookings

### Recommendation
COOX should implement a tiered geo-blocking system:

| Tier | Criteria | Action |
|------|----------|--------|
| HIGH | Cluster zones with 10+ failed bookings | Hard-block — prevent booking placement |
| MEDIUM | Cluster zones with 5-9 failed bookings | Soft-block — show service availability warning at checkout |
| LOW | Cluster zones with <5 failed bookings | Monitor — re-evaluate quarterly |
| ISOLATED | Non-clustered locations with known pincodes | Pincode-level blocking where repeat patterns emerge |

### Business Impact
- Reduction in refund rates through proactive area blocking
- Improved operational efficiency by eliminating unserviceable order processing
- Better customer experience — customers are informed upfront instead of facing cancellation
- Reduced revenue leakage from processing fees on refunded orders